In [1]:
!pip install -U langchain
!pip install -U langchain-cohere
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-chroma
!pip install -U pypdf
!pip install -U chromadb

  Using cached langchain_cohere-0.6.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached cohere-5.21.1-py3-none-any.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 9.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 53.7 MB/s eta 0:00:00a 0:00:01
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached langchain_classic-1.0.8-py3-none-any.whl (1.0 MB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
  Attempting uninstall: requests
    Found existing installation:

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
key = user_secrets.get_secret("coherekey")

----

## **Part 1 — Test the RAG with Your Own CV**

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/kaggle/input/datasets/islamohamed10/my-cv-v2/CV.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

print(documents[0].page_content[:500])

Number of pages: 2
Islam Mohamed
♂¶ap-¶arker-altCairo, Egypt✉10islammohamed01@gmail.com♂phone-alt+20 114 468 2583/linkedin-inIslam Mohamed
/githubislam0114
Summary
Computer and Communication Engineering student at Shoubra Engineering with a strong pas-
sion for Artificial Intelligence and Machine Learning. Actively building skills in machine learning,
deep learning, data preprocessing, and data analysis through intensive training programs and cer-
tifications from IBM, Microsoft, NTI, ITI, DEPI, and DataCamp. Know


In [6]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

In [8]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="pdf_rag"
)

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

In [10]:
query = "What is this document about?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
Islam Mohamed
♂¶ap-¶arker-altCairo, Egypt✉10islammohamed01@gmail.com♂phone-alt+20 114 468 2583/linkedin-inIslam Mohamed
/githubislam0114
Summary
Computer and Communication Engineering student at Shoubra Engineering with a strong pas-
sion for Artificial Intelligence and Machine Learning. Actively building skills in machine learning,
deep learning, data preprocessing, and data analysis through intensive training programs and cer-
tifications from IBM, Microsoft, NTI, ITI, DEPI, and DataCamp. Known for being organized,
disciplined with time, and a critical thinker who enjoys learning and exploring new AI concepts.
Currently seeking internship or junior roles to gain practical experience, apply theoretical knowl-
edge, and grow into a distinguished professional in the AI field.
Education
Metadata: {'page': 0, 'title': "Islam Mohamed's CV", 'producer': 'pdfTeX-1.40.27', 'creationdate': '2026-06-12T20:06:07+00:00', 'author': 'Islam Mohamed', 'source': '/kaggle/input/data

In [11]:
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)


In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

In [13]:
from langchain_core.runnables import RunnablePassthrough

In [14]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [15]:
question = "•	What are my main technical skills?"

response = rag_chain.invoke(question)

print(response.content)

Your main technical skills include:

- **Programming**: Python, Pandas, NumPy, Matplotlib, Seaborn, SciKit-learn, TensorFlow, Keras, PyTorch  
- **Machine Learning**: Supervised and Unsupervised Learning Algorithms, Model Training and Evaluation, Feature Selection, Hyperparameter Tuning  
- **Deep Learning**: Neural Networks, CNN, RNN, Transformers  
- **Data Science**: Data Preprocessing, EDA, Statistical Analysis, Data Visualization  
- **Deployment Tools**: Git, GitHub, Streamlit


In [16]:
question = "•	What machine learning experience do I have?"

response = rag_chain.invoke(question)

print(response.content)

Based on the context provided, your machine learning experience includes:

1. **Supervised and Unsupervised Learning**: Developed and fine-tuned models using algorithms like Random Forest, with a focus on feature selection and hyperparameter tuning to improve performance.  
2. **Deep Learning**: Explored architectures such as CNN (Convolutional Neural Networks) and RNN (Recurrent Neural Networks), and gained foundational experience in Transformers.  
3. **Model Training and Evaluation**: Performed data preprocessing, feature selection, and model fine-tuning to ensure high accuracy and explainability of predictions.  
4. **Projects**:  
   - Built a Random Forest model for **Bank Customer Churn Prediction**, achieving robust performance on imbalanced datasets.  
   - Developed a **Bilingual Semantic Chatbot** using a RAG pipeline with Sentence-BERT and Google Gemini API.  
5. **Tools and Frameworks**: Utilized Python, Scikit-learn, TensorFlow, Keras, PyTorch, and Streamlit for machine l

In [17]:
question = "•	What projects are mentioned in my CV?"

response = rag_chain.invoke(question)

print(response.content)

The projects mentioned in your CV are:

1. **Moshrif – Smart Attendance & University ERP System**: Built the core AI vision engine for a 4-layer microservices ERP system to fully automate university attendance.  
2. **Book Recommendation System**: Utilized 35MB ratings datasets to deliver personalized book suggestions based on user behavior.  
3. **Bilingual Semantic Chatbot**: Built a RAG pipeline using paraphrase-multilingual-MiniLM-L12-v2 and Google Gemini API.  
4. **Bank Customer Churn Project**: Developed and deployed a Random Forest model to predict bank customers at risk of churn, with an interactive Streamlit and Power BI dashboard for real-time insights.


In [18]:
question = "•	What programming languages do I know?"

response = rag_chain.invoke(question)

print(response.content)

Based on the context provided, the programming languages you know include:

- **Python** (extensively used across various projects and experiences)
- **React.js** (mentioned in the "Data Integration" project)

No other programming languages are explicitly listed in the context.


In [19]:
question = "•	What training or internships have I completed?"

response = rag_chain.invoke(question)

print(response.content)

Based on the context provided, you have completed the following training and internships:

1. **Data Science and Machine Learning Trainee, DEPI** (Dec 2025 – Jul 2026)  
2. **AI Summer Training – ITI** (Aug 2025)  
3. **Machine Learning – NTI** (Aug 2025)  
4. **AI & Machine Learning – Sprints x Microsoft** (Oct 2025)  
5. **IBM AI Engineer Certificate – Coursera** (Jan 2026)  
6. **AI & Data Science – DEPI** (May 2025)  

Additionally, you are currently seeking internship or junior roles to gain further practical experience.


----

## **Part 2 — Ask Questions About a Book**

In [21]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
key = user_secrets.get_secret("coherekey")

In [22]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/kaggle/input/datasets/islamohamed10/python/python crash course.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

print(documents[0].page_content[:500])

Number of pages: 562
A HANDS-ON , PROJECT-BASED
INTRODUCTION TO PROGRAMMING
ERIC MATTHES
P Y THON
C R ASH COURSE
P Y THON
C R ASH COURSE
SHELVE IN:
PROGRAMMING LANGUAGES/
PYTHON
$39.95 ($45.95 CDN)
FAST!
LEARN PYTHON—
FAST!
LEARN PYTHON—
PYTHON CRASH COURSEPYTHON CRASH COURSEMATTHES
COVERS PYTHON 2 AND 3
Python Crash Course is a fast-paced, thorough intro-
duction to programming with Python that will have you 
writing programs, solving problems, and making things 
that work in no time. 
In the first half of the book


In [23]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)


In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400
)

chunks = text_splitter.split_documents(documents)

In [25]:
from langchain_chroma import Chroma
import time

# To avoid the 429 Too Many Requests error with Trial keys,
# we decrease batch size and increase sleep time to stay under token limits.
batch_size = 30
vectorstore = Chroma(collection_name="pdf_rag", embedding_function=embeddings)

for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    vectorstore.add_documents(batch)
    print(f"Processed {i + len(batch)} / {len(chunks)} chunks...")
    if i + batch_size < len(chunks):
        # Trial keys often require a significant pause to reset token-per-minute counters
        time.sleep(7)

Processed 30 / 806 chunks...
Processed 60 / 806 chunks...
Processed 90 / 806 chunks...
Processed 120 / 806 chunks...
Processed 150 / 806 chunks...
Processed 180 / 806 chunks...
Processed 210 / 806 chunks...
Processed 240 / 806 chunks...
Processed 270 / 806 chunks...
Processed 300 / 806 chunks...
Processed 330 / 806 chunks...
Processed 360 / 806 chunks...
Processed 390 / 806 chunks...
Processed 420 / 806 chunks...
Processed 450 / 806 chunks...
Processed 480 / 806 chunks...
Processed 510 / 806 chunks...
Processed 540 / 806 chunks...
Processed 570 / 806 chunks...
Processed 600 / 806 chunks...
Processed 630 / 806 chunks...
Processed 660 / 806 chunks...
Processed 690 / 806 chunks...
Processed 720 / 806 chunks...
Processed 750 / 806 chunks...
Processed 780 / 806 chunks...
Processed 806 / 806 chunks...


In [26]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

In [27]:
query = "What is this document about?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
xxvi   Contents in Detail
IDLE  .  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on Linux . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on OS X  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on Windows  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
Customizing IDLE Settings  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
Emacs and vim  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
C 
gettIng helP 499
First Steps  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 499
Try It Again  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

In [28]:
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)


In [29]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

In [30]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [31]:
question = "what is the list"

response = rag_chain.invoke(question)

print(response.content)

A list is a collection of items in a particular order. You can put anything you want into a list, such as the letters of the alphabet, the digits from 0–9, or the names of all the people in your family. Lists are one of Python’s most powerful features, allowing you to store sets of information in one place, whether you have just a few items or millions of items.


In [32]:
question = "what is for loop?"

response = rag_chain.invoke(question)

print(response.content)

A **for loop** in Python is a control flow statement that allows you to iterate over a sequence (such as a list, tuple, dictionary, set, or string) and execute a block of code for each item in the sequence. It automates repetitive tasks by letting Python manage the iteration internally, so you don't have to manually retrieve each item from the list or change the code when the list's length changes.

For example, in the context provided, a for loop is used to print each magician's name from a list:

```python
magicians = ['alice', 'david', 'carolina']
for magician in magicians:
    print(magician)
```

Here, the for loop iterates over each name in the `magicians` list, assigns it to the variable `magician`, and then prints the name. This process repeats for every item in the list, making it efficient and scalable for lists of any size.


In [33]:
question = "who is mohamed salah?"

response = rag_chain.invoke(question)

print(response.content)

I don't know. The context provided does not contain any information about Mohamed Salah. It appears to be a resume or profile for a person named Islam Mohamed, who is a Computer and Communication Engineering student with a focus on Artificial Intelligence and Machine Learning. The rest of the context includes Python code examples and explanations related to programming concepts such as lists, slicing, and loops.


In [34]:
question = "what are the data types in python?"

response = rag_chain.invoke(question)

print(response.content)

Based on the provided context, the data types in Python mentioned are:

1. **Strings**: A series of characters enclosed in single or double quotes.  
2. **Integers**: Whole numbers without decimal points.  
3. **Floats**: Numbers with decimal points.  

These are the primary data types discussed in the context.


In [35]:
question = "how to define a function?"

response = rag_chain.invoke(question)

print(response.content)

To define a function in Python, you use the `def` keyword followed by the function name and parentheses `()`. Inside the parentheses, you can include parameters if the function requires any input. The function definition ends with a colon `:`. The body of the function, which contains the code to be executed, is indented below the definition line.

Here’s the basic structure:

```python
def function_name(parameters):
    """Docstring: Describe what the function does."""
    # Function body: Code to be executed
    pass
```

**Example from the context:**

```python
def greet_user():
    """Display a simple greeting."""
    print("Hello!")
```

In this example:
- `def` is the keyword to define a function.
- `greet_user` is the name of the function.
- The parentheses `()` are empty because this function doesn't require any parameters.
- The docstring `"""Display a simple greeting."""` describes what the function does.
- The body of the function contains the `print("Hello!")` statement.


In [36]:
question = "what is object oriented programming?"

response = rag_chain.invoke(question)

print(response.content)

Object-oriented programming (OOP) is one of the most effective approaches to writing software. In OOP, you write classes that represent real-world things and situations, and you create objects based on these classes. When you write a class, you define the general behavior that a whole category of objects can have. When you create individual objects from the class, each object is automatically equipped with the general behavior, and you can then give each object unique traits. OOP allows you to model real-world situations effectively and helps you understand the logic behind your code, not just line by line, but also the bigger concepts behind it. It also facilitates collaboration among programmers by providing a common logical framework for writing code.


### Chunking Experiment
Testing a different chunk size (chunk_size=800, chunk_overlap=100) and observing the retrieved context.

In [37]:
text_splitter_exp = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks_exp = text_splitter_exp.split_documents(documents)

# Create a temporary vectorstore for the experiment
vectorstore_exp = Chroma.from_documents(chunks_exp[:50], embeddings, collection_name="pdf_rag_exp")

retriever_exp = vectorstore_exp.as_retriever(search_kwargs={"k": 5})

query_exp = "what is the list"
retrieved_docs_exp = retriever_exp.invoke(query_exp)

print("Retrieved chunks for chunk_size=800:")
for i, doc in enumerate(retrieved_docs_exp):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content)

Retrieved chunks for chunk_size=800:

--- Document 1 ---
Exercise 2-11: Zen of Python ................................... 36
Summary  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 36
3 
IntroduCIng lIsts 37
What Is a List? . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 37
Accessing Elements in a List  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 38
Index Positions Start at 0, Not 1  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 39
Using Individual Values from a List  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 39
Exercise 3-1: Names ......................................... 40

--- Document 2 ---
Exercise 3-1: Names ......................................... 40
Exercise 3-2: Greetings ....................................... 40
Exercise 3-3: Your Own List ..................

----

## **Part 3 — Multi-PDF RAG**

In [38]:
from langchain_community.document_loaders import PyPDFLoader
import os

# Define the list of PDF files you want to load
pdf_files = ["/kaggle/input/datasets/islamohamed10/data-structure/Part 2 Linked List.pdf", "/kaggle/input/datasets/islamohamed10/data-structure/Part 3  Stack.pdf","/kaggle/input/datasets/islamohamed10/data-structure/Part 4 Queue.pdf","/kaggle/input/datasets/islamohamed10/data-structure/Part 5 Recursion.pdf"]

all_documents = []

for pdf_path in pdf_files:
    if os.path.exists(pdf_path):
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        all_documents.extend(docs)
        print(f"Loaded {pdf_path}: {len(docs)} pages")
    else:
        print(f"Warning: File not found at {pdf_path}")

print(f"\nTotal documents loaded: {len(all_documents)}")

Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 2 Linked List.pdf: 29 pages
Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 3  Stack.pdf: 18 pages
Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 4 Queue.pdf: 27 pages
Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 5 Recursion.pdf: 14 pages

Total documents loaded: 88


In [39]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)


In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split all loaded documents into chunks
chunks = text_splitter.split_documents(all_documents)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 113


In [41]:
from langchain_chroma import Chroma
import time

# To avoid the 429 Too Many Requests error with Trial keys,
# we decrease batch size and increase sleep time to stay under token limits.
batch_size = 30
vectorstore = Chroma(collection_name="pdf_rag", embedding_function=embeddings)

for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    vectorstore.add_documents(batch)
    print(f"Processed {i + len(batch)} / {len(chunks)} chunks...")
    if i + batch_size < len(chunks):
        # Trial keys often require a significant pause to reset token-per-minute counters
        time.sleep(3)

Processed 30 / 113 chunks...
Processed 60 / 113 chunks...
Processed 90 / 113 chunks...
Processed 113 / 113 chunks...


In [42]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

In [43]:
query = "What is this document about?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
xxvi   Contents in Detail
IDLE  .  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on Linux . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on OS X  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on Windows  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
Customizing IDLE Settings  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
Emacs and vim  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
C 
gettIng helP 499
First Steps  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 499
Try It Again  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

In [44]:
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)


In [45]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

In [46]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [47]:
question = "what is the linkedlist?"

response = rag_chain.invoke(question)

print(response.content)

Based on the provided context, a **LinkedList** is a data structure composed of a sequence of elements called **links** or **nodes**. Each link contains data and a reference (or pointer) to the next link in the sequence. The **LinkList** class, for example, contains a reference to the first link (`first`) in the list. 

Here are the key characteristics of a LinkedList from the context:

1. **Link Class**: Each link in the list contains data (e.g., `iData`, `dData`, or `dData`) and a reference to the next link (`next`).
2. **LinkList Class**: Manages the list by holding a reference to the first link (`first`). It includes methods like `isEmpty()` to check if the list is empty.
3. **FirstLastList Class**: An extension of the basic LinkedList that also maintains a reference to the last link (`last`), allowing for efficient insertion and deletion at both ends of the list.

In summary, a LinkedList is a linear collection of nodes where each node points to the next node in the sequence.


In [48]:
question = "what is the stack?"

response = rag_chain.invoke(question)

print(response.content)

A stack is a LIFO (Last In, First Out) data structure that allows access to only one data item: the last item inserted. If you remove this item, you can access the next-to-last item inserted, and so on.


In [49]:
question = "who is mohamed salah?"

response = rag_chain.invoke(question)

print(response.content)

I don't know. The context provided does not contain any information about Mohamed Salah. It appears to be a resume or profile for a person named Islam Mohamed, who is a Computer and Communication Engineering student with a focus on Artificial Intelligence and Machine Learning. The rest of the context includes Python code examples and explanations related to programming concepts such as lists, slicing, and loops.


## **Bonus**

In [62]:
import os
import time
import re

print("Killing old processes...")
os.system("pkill -f streamlit")
os.system("pkill -f cloudflared")
time.sleep(2)

print("Starting new Streamlit app...")
os.system("python -m streamlit run /kaggle/input/datasets/islamohamed10/streamlit-v5/app.py --server.enableCORS false --server.enableXsrfProtection false &")

print("Downloading and Starting Cloudflare Tunnel...")
os.system("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
os.system("chmod +x cloudflared-linux-amd64")

os.system("rm -f tunnel.log")
os.system("nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 > tunnel.log 2>&1 &")

print("Waiting for the tunnel to connect (5-10 seconds)...")
time.sleep(8)

with open("tunnel.log", "r") as f:
    logs = f.read()
    url = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)
    if url:
        print("your url", url.group())
    else:
        print("!grep -o 'https://.*\.trycloudflare\.com' tunnel.log")


<>:29: SyntaxWarning: invalid escape sequence '\.'
<>:29: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_129/74538982.py:29: SyntaxWarning: invalid escape sequence '\.'
  print("!grep -o 'https://.*\.trycloudflare\.com' tunnel.log")


Killing old processes...
  Stopping...
  Stopping...
  Stopping...
Starting new Streamlit app...
Waiting for the tunnel to connect (5-10 seconds)...




2026-08-11 12:37:39.319 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.29.15.33:8501

your url https://enable-scheduling-thread-aaa.trycloudflare.com


/kaggle/input/datasets/islamohamed10/streamlit-v5/app.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
